# Ticket 4: idempotent period loader (2024-01)

Scope: `company_code = 1000, fiscal_year = 2024, fiscal_period = 1`.
What `fact_gl_line` needs to handle, checked before writing
`src/load_fact.py`. Queries run through `duckdb` against
`warehouse.duckdb`, the same SQL the loader runs.

In [1]:
import duckdb # type: ignore

con = duckdb.connect("../warehouse.duckdb", read_only=True)
SCOPE = "company_code = 1000 AND fiscal_year = 2024 AND fiscal_period = 1"
con.execute(f"SELECT COUNT(*), COUNT(DISTINCT document_id) FROM stg_gl WHERE {SCOPE}").fetchall()

[(13142, 3518)]

**13,142 rows, 3,518 distinct documents** in scope.

In [2]:
# document_type distribution: which of the 12 counted types actually show up,
# and is anything outside the catalog (would need unmapped_doc_type)
con.execute(f"""
    SELECT document_type, COUNT(*), COUNT(DISTINCT document_id)
    FROM stg_gl WHERE {SCOPE} GROUP BY 1 ORDER BY 2 DESC
""").fetchall()

[('DR', 4248, 1183),
 ('KR', 4168, 1023),
 ('SA', 2903, 822),
 ('HR', 1348, 328),
 ('AA', 419, 142),
 ('IC', 31, 15),
 ('OPENING_BALANCE', 17, 1),
 ('WE', 4, 2),
 ('WL', 4, 2)]

`DR 4248/1183, KR 4168/1023, SA 2903/822, HR 1348/328, AA 419/142,
IC 31/15, OPENING_BALANCE 17/1, WE 4/2, WL 4/2`. Every type present is
in the counted catalog except `OPENING_BALANCE`, which is excluded from
in-period totals by rule, not because it's unrecognized. No
`unmapped_doc_type` case shows up in this period, but the loader still
needs to handle one.

In [3]:
# unbalanced documents: debit - credit != 0 at the document grain
con.execute(f"""
    SELECT document_id, document_type, ROUND(SUM(debit_amount) - SUM(credit_amount), 2) AS gap,
           bool_or(is_fraud) AS any_fraud, bool_or(is_anomaly) AS any_anomaly
    FROM stg_gl WHERE {SCOPE} GROUP BY 1, 2 HAVING gap != 0
""").fetchall()

[('6a00ef40-e8ec-4c9d-89b1-2c3440301a67', 'DR', 90.0, False, True)]

**Exactly 1 unbalanced document** in this period. Per
`docs/business-rules.md`: load to staging, never publish. Real test
case for the `unbalanced_document` exclusion, small enough to hand-check
after the loader runs.

In [4]:
# duplicate grain: company + document + line + year + period appearing more than once
con.execute(f"""
    SELECT COUNT(*) FROM (
        SELECT company_code, document_id, line_number, fiscal_year, fiscal_period, COUNT(*) n
        FROM stg_gl WHERE {SCOPE} GROUP BY 1,2,3,4,5 HAVING n > 1
    )
""").fetchall()

[(0,)]

**Zero duplicate-grain groups** in this period. Clean this round, but the
`duplicate_source` blocking check still belongs in the loader (or #5's
gate), it just has nothing to catch here.

In [5]:
# flagged-row volume, since these flow through unchanged (ADR-0003) and must not get filtered
con.execute(f"""
    SELECT SUM(is_post_close::int), SUM(is_fraud::int), SUM(is_anomaly::int)
    FROM stg_gl WHERE {SCOPE}
""").fetchall()

[(200, 326, 516)]

`is_post_close=200, is_fraud=326, is_anomaly=516` rows. All three must
still load into `fact_gl_line`: `is_post_close` is excluded from the
in-period close *total* but not from the table, `is_fraud`/`is_anomaly`
stay untouched either way (ADR-0003).

## What I've got

1. `OPENING_BALANCE` rows (17 of them, 1 document) stay in `fact_gl_line`
   with a flag, not excluded from the table. Real volume there, not
   zero.
2. The 1 unbalanced document is a real, small, checkable case, not a
   theoretical rule. The loader's `unbalanced_document` exclusion has
   something to actually exclude and verify against.
3. No `unmapped_doc_type` case exists in this period, so that path
   can't be proven by this period alone. Flagged as a gap in the
   mission's deterministic checks: the column/logic exists, but
   nothing here proves it fires correctly.